# Mini Project: Sales Report Cleaner

**The scenario:** Your colleague has sent you a CSV export from the CRM. It has inconsistent formatting, missing values, and amounts stored as text (with a € symbol). Your job is to clean it with Python and produce a tidy summary by region.

**What you will build:** A Python script that:
1. Opens and reads the raw CSV file
2. Cleans each row (fixes formats, skips bad data)
3. Calculates totals per region
4. Exports a clean summary CSV

---

## Step 1 — Open and inspect the file

Before cleaning anything, let's look at what we're dealing with.

**New concept: `import`**  
Python comes with built-in tools called *modules*. We need to `import` them before using them.  
The `csv` module knows how to read CSV files properly (handling commas inside fields, quotes, etc.).

In [ ]:
import csv

# open() opens the file. 'r' means we want to READ it (not write to it).
# encoding='utf-8-sig' handles the € symbol correctly.
with open('sales_data_raw.csv', 'r', encoding='utf-8-sig') as file:
    reader = csv.DictReader(file)   # DictReader reads each row as a dictionary
    
    print('--- Column names found in the file ---')
    print(reader.fieldnames)
    print()
    
    print('--- First 5 rows ---')
    for i, row in enumerate(reader):
        print(dict(row))
        if i == 4:          # stop after 5 rows (index 0 to 4)
            break

**What do you notice?** Look at the output above. Can you spot:
- Rows where the region is in UPPERCASE, or lowercase, or mixed?
- The `amount` column — it's a number, but it has a `€` sign. What type is it?
- Any rows where `date` or `region` appears to be missing?

These are the problems we will fix in the next steps.

---

## Step 2 — Clean a single row (practice before the loop)

Before processing ALL rows, let's practice the cleaning operations on a single example.  
This is a common professional approach: solve a small version of the problem first.

**New concepts:**
- **Variables** store a value under a name so you can use it later
- **Strings** are text values — always in quotes
- **`str.lower()`** converts a string to lowercase
- **`str.replace()`** swaps one piece of text for another
- **`float()`** converts a string like `'1200.00'` into a real number `1200.0`

In [ ]:
# --- Problem 1: region has inconsistent capitalisation ---
region_raw = 'NORTH'          # this is a string variable
region_clean = region_raw.lower()   # .lower() returns a new lowercase string
print('Raw region:', region_raw)
print('Clean region:', region_clean)
print()

# --- Problem 2: amount is stored as text with a € symbol ---
amount_raw = '€1200.00'       # this looks like a number, but Python sees it as text

# Step 1: remove the € symbol with .replace()
# .replace('old text', 'new text') — replacing with '' means deleting it
amount_no_symbol = amount_raw.replace('€', '')
print('After removing €:', amount_no_symbol)

# Step 2: convert the text '1200.00' into a real number
amount_number = float(amount_no_symbol)
print('As a number:', amount_number)
print('Now we can do maths:', amount_number * 2)

**Exercise:** In the cell below, try converting `'€342.75'` into a number and multiplying it by 3.

In [ ]:
# Your code here


---
## Step 3 — Loop through all rows and skip bad data

Now we apply our cleaning to every row automatically.  

**New concepts:**
- **`for` loop** — repeats a block of code for each item in a sequence
- **`if` / `else`** — makes a decision: "if this condition is true, do this; otherwise, do that"
- **`list`** — an ordered collection of items, like a column in a spreadsheet
- **`list.append()`** — adds an item to the end of a list

In [ ]:
import csv

clean_rows = []     # an empty list — we will fill it with clean rows
skipped = 0         # a counter for rows we skip

with open('sales_data_raw.csv', 'r', encoding='utf-8-sig') as file:
    reader = csv.DictReader(file)
    
    for row in reader:      # loop: Python runs the indented block once per row
        
        # --- GUARD CLAUSES: skip rows with missing critical data ---
        # If a required field is empty, we skip this row entirely
        if not row['date'] or not row['region']:
            print(f"Skipping row — missing date or region: {dict(row)}")
            skipped += 1
            continue        # 'continue' jumps to the next iteration of the loop
        
        # --- CLEAN the region ---
        region = row['region'].lower().strip()   # .strip() removes accidental spaces
        
        # --- CLEAN the amount ---
        amount_text = row['amount'].replace('€', '').strip()
        
        # Protect against amounts that can't be converted
        if not amount_text:
            print(f"Skipping row — missing amount for {row['salesperson']}")
            skipped += 1
            continue
        
        amount = float(amount_text)
        
        # --- CLEAN quantity (may also be missing) ---
        quantity_text = row['quantity'].strip()
        if quantity_text:
            quantity = int(quantity_text)
        else:
            quantity = 0    # treat missing quantity as 0
        
        # --- Build a clean row as a dictionary ---
        clean_row = {
            'date':        row['date'],
            'region':      region,
            'salesperson': row['salesperson'],
            'product':     row['product'],
            'amount':      amount,
            'quantity':    quantity
        }
        
        clean_rows.append(clean_row)   # add the clean row to our list

print(f"\nDone. {len(clean_rows)} rows cleaned, {skipped} rows skipped.")
print("\nFirst 3 clean rows:")
for row in clean_rows[:3]:
    print(row)

**What just happened?**
- The `for` loop ran the indented block once for each of the 30 rows in the file
- The `if not row['date']` check caught any row missing a date — like an `if` in Excel but written once for all rows
- We now have `clean_rows`: a list of dictionaries, each representing one valid, cleaned sale

---

## Step 4 — Summarise by region

This is the equivalent of a pivot table in Excel — but written in Python.  
We want one row per region with: total sales, number of transactions, average deal size.

**New concept: `dictionary`**  
A dictionary stores data as **key → value** pairs. Think of it like a lookup table.  
```python
totals = {'north': 5200.0, 'south': 3100.5}
totals['north']   # → 5200.0
```

In [ ]:
# We will build a dictionary where:
#   key   = region name (e.g. 'north')
#   value = another dictionary with totals for that region

summary = {}   # empty dictionary

for row in clean_rows:
    region = row['region']
    amount = row['amount']
    
    # If this region hasn't appeared yet, create an entry for it
    if region not in summary:
        summary[region] = {
            'total_sales':    0.0,
            'num_transactions': 0
        }
    
    # Add this row's data to the region's running totals
    summary[region]['total_sales']     += amount   # += means "add to itself"
    summary[region]['num_transactions'] += 1

# Calculate the average deal size for each region
for region in summary:
    total = summary[region]['total_sales']
    count = summary[region]['num_transactions']
    summary[region]['avg_deal_size'] = round(total / count, 2)
    summary[region]['total_sales']   = round(total, 2)

# Print the summary in a readable way
print(f"{'Region':<10} {'Total Sales':>14} {'Transactions':>14} {'Avg Deal':>12}")
print('-' * 54)
for region, data in sorted(summary.items()):   # sorted() puts regions in alphabetical order
    print(f"{region:<10} {data['total_sales']:>14.2f} {data['num_transactions']:>14} {data['avg_deal_size']:>12.2f}")

**What just happened?**  
The `summary` dictionary now contains aggregated data — just like a pivot table, but built from logic you can read, modify, and reuse.

---

## Step 5 — Export the summary to a CSV file

The payoff: we write our results to a file that anyone can open in Excel.

**New concept: writing files**  
We use `open()` again, but with `'w'` instead of `'r'` — `'w'` means *write*.  
The `with` keyword ensures the file is closed properly when we're done, even if something goes wrong.

In [ ]:
output_filename = 'sales_summary_by_region.csv'

# Define the column names for our output file
fieldnames = ['region', 'total_sales', 'num_transactions', 'avg_deal_size']

with open(output_filename, 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    
    writer.writeheader()   # write the column names as the first row
    
    for region in sorted(summary.keys()):
        writer.writerow({
            'region':           region,
            'total_sales':      summary[region]['total_sales'],
            'num_transactions': summary[region]['num_transactions'],
            'avg_deal_size':    summary[region]['avg_deal_size']
        })

print(f"File saved: {output_filename}")
print("Open it in Excel to see the results!")

---
## Step 6 — Reflect and extend

You've built a complete data pipeline in Python. Let's think about what that means.

**Discussion questions:**
1. How long would it take to do this manually in Excel for 30 rows? What about 300,000 rows?
2. What happens if next month's file has the same messy format? How much work is it to run this script again?
3. Where could this break? What assumptions did we make about the data?

**Extension challenges** (if you finish early):

Try modifying the script to also produce a summary **by salesperson** instead of by region.  
Hint: you only need to change one thing in Step 4.

In [ ]:
# Extension: summary by salesperson
# Copy your Step 4 code here and modify it


---
## Concepts covered in this project

| Concept | Where you used it |
|---|---|
| `import` | Loading the `csv` module |
| Variables | Storing region, amount, quantity |
| Strings | Text values like region names |
| `str.lower()`, `str.replace()`, `str.strip()` | Cleaning messy text |
| `float()`, `int()` | Converting text to numbers |
| `if` / `else` | Skipping bad rows, handling missing data |
| `for` loop | Processing every row automatically |
| List & `append()` | Collecting clean rows |
| Dictionary | Storing and looking up totals by region |
| File reading & writing | Opening the raw CSV, saving the output |